<a href="https://colab.research.google.com/github/akrishnapriya10/CeloFact/blob/main/exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PDF Processing via LLMaaS

## 0. Setting your Name and Email

Please starting by putting your name and email in the following variables - please stick to the required format i.e. NAME_SURNAME

In [ ]:
# WRITE YOUR NAME_SURNAME HERE, AS WELL AS YOUR EMAIL WITH WHICH YOU LOGGED IN INTO CELONIS
MY_NAME = 'KRISHNAPRIYA A'
MY_EMAIL = 'akrishnapriya10@gmail.com'

## 1. Extract information from Invoices

This is the section you will need to fill in. Your code should create the following
- **a pandas dataframe called df that includes the extracted information**.
- **the dataframe should contain a column called 'po_reference' that contains the reference to the PO**
- **the values in the column 'po_reference' should be a 11-char long strings. Use left padding with zeros where needed.**

In [ ]:
import os
# ADD YOU CODE HERE
!apt-get update
!apt-get install -y poppler-utils
!pip install transformers accelerate qwen-vl-utils pillow pdf2image pandas torch

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,849 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [101 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:13 https://ppa.launchpadcontent.net/ubuntug

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
import os
os.makedirs("/content/invoices", exist_ok=True)

In [ ]:
import os
import re
import glob
import json
import base64
import io
import pandas as pd
from PIL import Image
from pdf2image import convert_from_path
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# ---- CONFIG -----------------------------------------------------
INPUT_FOLDER = "/content/invoices"      # <-- change to wherever you uploaded files
PO_REFERENCE_LENGTH = 11
MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"  # free, open-weight, no API key needed
# -------------------------------------------------------------------

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(
    MODEL_NAME, min_pixels=256*28*28, max_pixels=1024*28*28
)

EXTRACTION_PROMPT = """Look at this invoice image and extract exactly these seven fields:

- vendor_name
- vendor_address
- payment_terms
- invoice_value (just the number, no currency symbol)
- company_code
- po_reference (digits only, as shown)
- invoice_id

Respond with ONLY a raw JSON object with those seven keys, no markdown \nformatting, no code fences, no explanation. If a field is not present, \nuse an empty string for that field."""


def file_to_image_path(filepath: str) -> str:
    ext = os.path.splitext(filepath)[1].lower()
    if ext == ".pdf":
        pages = convert_from_path(filepath, dpi=100)  # Reduced DPI from 200 to 100
        png_path = filepath.rsplit(".", 1)[0] + "_page1.png"
        pages[0].save(png_path)  # first page only
        return png_path
    return filepath


def encode_image_to_base64(image_path: str) -> str:
    """Encode an image file (PNG/JPEG, or a PDF's converted first page) to base64."""
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def extract_fields_with_vision(filepath: str) -> dict:
    image_path = file_to_image_path(filepath)

    # base64-encode the same image used for extraction, so what's displayed
    # in Celonis matches what the model actually read
    image_base64 = encode_image_to_base64(image_path)

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image_path},
            {"type": "text", "text": EXTRACTION_PROMPT},
        ],
    }]

    text_prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_prompt], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(model.device)

    generated_ids = model.generate(**inputs, max_new_tokens=512)
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    raw_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0].strip()

    # strip markdown code fences if the model added them anyway
    raw_text = re.sub(r"^```(?:json)?|```$", "", raw_text, flags=re.MULTILINE).strip()

    try:
        fields = json.loads(raw_text)
    except json.JSONDecodeError:
        print(f"WARNING: could not parse JSON for {filepath}. Raw output:\n{raw_text}\n")
        fields = {k: "" for k in ["vendor_name", "vendor_address", "payment_terms",
                                   "invoice_value", "company_code", "po_reference", "invoice_id"]}

    fields["invoice_image_base64"] = image_base64
    return fields


def build_dataframe(folder: str) -> pd.DataFrame:
    filepaths = sorted(
        glob.glob(os.path.join(folder, "*.pdf")) +
        glob.glob(os.path.join(folder, "*.png")) +
        glob.glob(os.path.join(folder, "*.jpg")) +
        glob.glob(os.path.join(folder, "*.jpeg"))
    )

    records = []
    for path in filepaths:
        print(f"Processing {os.path.basename(path)}...")
        fields = extract_fields_with_vision(path)
        fields["source_file"] = os.path.basename(path)
        records.append(fields)

    df = pd.DataFrame(records)
    df["po_reference"] = (
        df["po_reference"].astype(str).str.replace(r"\D", "", regex=True)
        .str.zfill(PO_REFERENCE_LENGTH)
    )

    cols = ["invoice_id", "vendor_name", "vendor_address",
            "payment_terms", "invoice_value", "company_code", "po_reference",
            "invoice_image_base64"]
    return df[cols]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

In [ ]:
df = build_dataframe(INPUT_FOLDER)
print(df.to_string(index=False))

Processing  INV-2020-08-001247.pdf...
Processing  INV-2020-08-001247_page1.png...
Processing INV-2020-001.png...
Processing INV-2020-07-001853.pdf...
Processing INV-2020-07-001853_page1.png...
Processing INV-2021-001.png...
Processing INV-2021-002.png...
        invoice_id          vendor_name                                                                   vendor_address                                                                                                                                                           payment_terms invoice_value company_code po_reference                                                                                                                                                                                                                                                                                                                                                                                                                                 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Pushing Data back to Celonis

The below script will push your results into Celonis, it may take **up to 5 minutes** for the results to show up in Celonis frontend. Please refrain from triggering the below script multiple times in a row.

In [ ]:
%run push.ipynb

Successfully pushed file to Celonis


FileNotFoundError: [Errno 2] No such file or directory: 'Dataframes/DOC_EXTRACTION_KRISHNAPRIYA A_akrishnapriya10@gmail.com.csv'

In [ ]:
import base64
import os
from pdf2image import convert_from_path
from PIL import Image
import io

def encode_file(filepath):
    if filepath.lower().endswith('.pdf'):
        pages = convert_from_path(filepath, dpi=150)
        img = pages[0]
        buffer = io.BytesIO()
        img.save(buffer, format='PNG')
        return base64.b64encode(buffer.getvalue()).decode('utf-8')
    else:
        with open(filepath, "rb") as f:
            return base64.b64encode(f.read()).decode('utf-8')

invoice_folder = "/content/invoices"

# build a base64 value for each file, keyed by filename
image_data = {}
for filename in os.listdir(invoice_folder):
    filepath = os.path.join(invoice_folder, filename)
    if os.path.isfile(filepath):
        image_data[filename] = encode_file(filepath)

# then, when building each row of your dataframe, match the invoice's
# filename to pull in the right base64 string, e.g.:
# row['invoice_image_base64'] = image_data[row['source_filename']]

# add as a new field alongside your other extracted values
row['invoice_image_base64'] = encode_image(invoice_filepath)

NameError: name 'invoice_filepath' is not defined